In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

# Configure logger to ignore everything to avoid cluttering the output
import logging
logging.getLogger().setLevel(logging.WARNING)

import dotenv # load env vars from .env
dotenv.load_dotenv()

from openai import OpenAI
import dotenv  
import os   

dotenv.load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

/Users/admin/repos/geneforge


# Direct Preference Optimization

#### Collect preferred and non-preferred response

In [ ]:
%%capture
import os
import json
from src.examples.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow

start_index = 0
num_runs = 5
all_dpo_pairs = []

os.environ["OPENAI_MODEL"] = "gpt-4.1-2025-04-14"  

run_name = "run" 
for run_number in range(start_index, start_index + num_runs):
    run_id = f"{run_name}_{run_number}"
    from src.library.cello_library import CelloLibrary
    library = CelloLibrary()
    library.select_library("Eco1C1G1T1")
    promoters = [part for part in library.get_ucf_data() if part.get("collection") == "parts" and part.get("type") == "promoter"]
    promoter_sequence = promoters[0].get("dnasequence")
    print(promoters[0])
    workflow = MaximizePromoterStrengthWorkflow(
        promoter_sequence=promoter_sequence,
        example_name="MaximizePromoterStrength",
    )
    
    DPO_OUTPUT_DIR = f"datasets/{workflow.example_name}_dpo"

    print(f"Running {run_id}")
    dpo_pairs = workflow.run_generate_preference_pair_on_tool_failures()
    print(f"Collected {len(dpo_pairs)} dpo pairs")
    
    all_dpo_pairs += dpo_pairs
    os.makedirs(f"{DPO_OUTPUT_DIR}/{workflow.model}/{run_id}", exist_ok=True)
    with open(f"{DPO_OUTPUT_DIR}/{workflow.model}/{run_id}/dpo_pairs.jsonl", "w") as f:
        for dpo_pair in dpo_pairs:
            f.write(json.dumps(dpo_pair) + "\n")
            
with open(f'{DPO_OUTPUT_DIR}/{workflow.model}/{run_name}_all_dpo_pairs.jsonl', "w") as f:
    for dpo_pair in all_dpo_pairs:
        f.write(json.dumps(dpo_pair) + "\n")

In [9]:
print(promoters[0])

{'collection': 'parts', 'type': 'promoter', 'name': 'pAmtR', 'dnasequence': 'CTTGTCCAACCAAATGATTCGTTACCAATTGACAGTTTCTATCGATCTATAGATAATGCTAGC'}


##### Upload the training file and run the fine-tuning job

In [ ]:
import json
from sklearn.model_selection import train_test_split

dpo_all_dpo_pairs_path = f"{DPO_OUTPUT_DIR}/{runner.model}/{run_name}_all_dpo_pairs.jsonl"
all_samples = [json.loads(line) for line in open(dpo_all_dpo_pairs_path)]

# split into train and test set
train_set_path = dpo_all_dpo_pairs_path.split("_all_dpo_pairs.jsonl")[0] + "_train.jsonl"
test_set_path = dpo_all_dpo_pairs_path.split("_all_dpo_pairs.jsonl")[0] + "_test.jsonl"
train_samples, test_samples = train_test_split(all_samples, test_size=0.2, random_state=42)

json.dump(train_samples, open(train_set_path, "w"))
json.dump(test_samples, open(test_set_path, "w"))
    
uploaded_file = client.files.create(
    file=open(train_set_path, "rb"),
    purpose="fine-tune",
)
uploaded_file.id

job = client.fine_tuning.jobs.create(
    training_file=uploaded_file.id,
    model="gpt-4.1-2025-04-14",
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {"beta": 0.1},
        },
    },
)

print(job)

In [ ]:
# Evaluate the original model. We can use the pregenerated chat histories. (TODO Split into test and train set)
from src.examples.agent.design_w_promoter_vars import score_run_from_directory

i = 0
scores_original = {}
for i in range(10):
    MODEL_NAME = "gpt-4.1-2025-04-14" # chat histories are stored under a model specific directory
    directory = f"outputs/chat_histories/{MODEL_NAME}/design_w_promoter_vars_dataset_{i}"
    score = score_run_from_directory(directory)
    scores_original[directory] = score

In [ ]:
# Run sessions with trained model (n=10)
TRAINED_MODEL_NAME_ID = "ft:gpt-4.1-2025-04-14:geneforge::BsWZyydP"

os.environ["OPENAI_MODEL"] = TRAINED_MODEL_NAME_ID

workflow = get_runner(max_rounds=25, max_attempts=3)
workflow.setup()
workflow.generate_chat_histories(output_dir="outputs/chat_histories", base_run_name="design_w_promoter_vars_dataset", num_runs=10, start_index=0)

scores_ft = scores_for_runs_from_directory(f"outputs/chat_histories/{MODEL_NAME}")